In [98]:
import os
from dotenv import load_dotenv
import json
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
import typing_extensions as typing

def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=google_api_key)

medicine_list = []
def medicine_tool(medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list

medicine_class={
    "name": "medicine_tool",
    "description":"gets the list of medicines",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}

tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_class"]
    }
}

# user prompt for all 
user_prompt=f"list of drug:\n"
user_prompt+=str([medicine_class])

# system prompt for drug drug classification 
system_prompt_ddc ={"text":"""you are an medical AI agent.Respond the reaction between the drugs to human. Classify the drug on the bases:
                1.Major:Highly clinically significant. Avoid combinations; the risk of the interaction outweighs the benefit.
                2.moderate:Moderately clinically significant. Usually avoid combinations; use it only under special circumstances.
                3.minor:Minimally clinically significant. Minimize risk; assess risk and consider an alternative drug, take steps to circumvent the interaction risk and/or institute a monitoring plan.
                4.Unknown:Unknown interaction information available./n
                answer in 1 word"""}

# system prompt for drug drug description
system_prompt_ddd ={"text":"you are an medical AI agent.Respond the reaction between the drugs to human when taken together in 100 words"}

# system prompt for drug food classification
system_prompt_dfc ={"text":"""You are a medical AI Agent. Classify on the bases:
                1.Major:Highly clinically significant. Avoid combinations; the risk of the interaction outweighs the benefit.
                2.Moderate:Moderately clinically significant. Usually avoid combinations; use it only under special circumstances.
                3.Minor:Minimally clinically significant. Minimize risk; assess risk and consider an alternative drug, take steps to circumvent the interaction risk and/or institute a monitoring plan.
                4.Unknown:Unknown interaction information available./n
                answer in 1 word"""}

# system prompt for drug food description
system_prompt_dfd ={"text":"""You are a medical AI Agent. Tell is there any food which can react with the drugs in the list in 100 words"""}

# system prompt for theraputic-duplication classification
system_prompt_tdc ={"text":"""You are a medical AI Agent.Classify on the bases:
                1.Major:Highly clinically significant. Avoid combinations; the risk of the interaction outweighs the benefit.
                2.Moderate:Moderately clinically significant. Usually avoid combinations; use it only under special circumstances.
                3.Minor:Minimally clinically significant. Minimize risk; assess risk and consider an alternative drug, take steps to circumvent the interaction risk and/or institute a monitoring plan.
                4.Unknown:Unknown interaction information available./n
                answer in 1 word"""}

# system prompt for theraputic-duplication description
system_prompt_tdd ={"text":"""You are a medical AI Agent.Tell is there any theraputic duplication in the list in 100 words"""}

# result of drug drug classification
def message_gemini_ddc(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_ddc,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

# result of drug drug description
def message_gemini_ddd(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_ddd,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

# result of drug food classification
def message_gemini_dfc(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_dfc,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

# result of drug food description
def message_gemini_dfd(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_dfd,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

# result of theraputic-duplication classification
def message_gemini_tdc(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_tdc,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

# result of theraputic duplication description
def message_gemini_tdd(*user_prompt):
    gemini = genai.GenerativeModel(
    'gemini-1.5-flash',
    system_instruction=system_prompt_tdd,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text

In [99]:
to_markdown(message_gemini_ddc("paracetamol","flexon"))

> Minor


In [91]:
to_markdown(message_gemini_ddd("paracetamol","flexon"))

> Combining paracetamol (acetaminophen) and ibuprofen (often found together in formulations like "paracetamol/flexon") generally enhances pain and fever relief compared to using either drug alone.  However, taking them together increases the risk of gastrointestinal side effects like nausea, stomach upset, and heartburn.  Liver damage is a potential, though rare, risk with excessive paracetamol doses, regardless of ibuprofen use. Always follow prescribed dosages and consult a doctor if you experience any adverse effects.  This information is not a substitute for professional medical advice.


In [92]:
to_markdown(message_gemini_dfc("paracetamol","flexon"))

> Minor


In [93]:
to_markdown(message_gemini_dfd("paracetamol","flexon"))

> Paracetamol (acetaminophen) and Flexon (containing ibuprofen) are common pain relievers.  Generally, no specific foods dramatically interact with them to cause significant harm.  However, excessive alcohol consumption while taking paracetamol can increase the risk of liver damage.  Grapefruit juice can potentially interact with some medications, though this is not consistently reported with paracetamol or ibuprofen.  Always consult a doctor or pharmacist if you have concerns about food-drug interactions, especially if you have pre-existing health conditions.


In [94]:
to_markdown(message_gemini_tdc("paracetamol","flexon"))

> Minor


In [95]:
to_markdown(message_gemini_tdd("paracetamol","flexon"))

> Yes, there's therapeutic duplication in the combination of paracetamol and flexon (assuming "flexon" refers to a muscle relaxant containing cyclobenzaprine or similar).  Both paracetamol (acetaminophen) and many muscle relaxants address pain, albeit through different mechanisms.  Taking both simultaneously may lead to increased side effects without a proportional increase in pain relief, thus representing a duplication of therapy.  A doctor should review this combination.
